In [0]:
"""
Local sanity check for etl/06_analytics/40_fare_efficiency_by_borough.sql

You don't have a Databricks warehouse to dry-run the actual SQL against
from your machine, so this reimplements the same CTE logic in pandas
against small, hand-built synthetic data where you already know the
right answer. It's not a substitute for running the real SQL once you
have warehouse access -- it's a fast way to catch a logic bug (wrong
direction on the rank, wrong eligibility filter, wrong DQ gate math)
before you commit.

Run: python test_fare_per_mile.py
"""

import pandas as pd

# ---------------------------------------------------------------
# Synthetic fact_taxi_trip rows. Each row is one already-aggregated
# "trip" record with the same columns the real Gold table has.
# Deliberately includes:
#   - a normal, clean trip in each borough
#   - a trip with a negative fare flag (should drop OUT of fare math,
#     but its trip_count must still be counted)
#   - a trip with a negative distance flag (same idea, for distance)
#   - a trip with distance == 0 (must be excluded from distance math
#     even though its flag says "not negative", since 0 makes the
#     ratio undefined)
#   - a trip whose pickup_zone_key has NO match in dim_taxi_zone
#     (tests the "rows_with_unresolved_borough" DQ gate)
# ---------------------------------------------------------------
fact_taxi_trip = pd.DataFrame([
    # zone_key, trip_count, trip_distance_miles, fare_amount_usd, neg_distance_flag, neg_fare_flag
    ["Z_MANHATTAN_1", 1, 5.0, 25.0, False, False],   # clean
    ["Z_MANHATTAN_1", 1, 2.0, 40.0, False, False],   # clean, expensive -> pushes Manhattan's $/mi up
    ["Z_BROOKLYN_1",  1, 10.0, 20.0, False, False],  # clean, cheap -> pushes Brooklyn's $/mi down
    ["Z_BROOKLYN_1",  1, 3.0, -5.0, False, True],    # bad fare: counted in trip_count, excluded from fare sum
    ["Z_QUEENS_1",    1, 8.0, 30.0, False, False],   # clean
    ["Z_QUEENS_1",    1, -1.0, 15.0, True, False],   # bad distance: counted in trip_count, excluded from distance sum
    ["Z_QUEENS_1",    1, 0.0, 12.0, False, False],   # zero distance: must ALSO be excluded even though flag is False
    ["Z_UNKNOWN_99",  1, 4.0, 20.0, False, False],   # zone_key with no match in dim_taxi_zone at all
], columns=[
    "pickup_zone_key", "trip_count", "trip_distance_miles",
    "fare_amount_usd", "negative_trip_distance_flag", "negative_fare_amount_flag",
])

dim_taxi_zone = pd.DataFrame([
    ["Z_MANHATTAN_1", "Manhattan"],
    ["Z_BROOKLYN_1", "Brooklyn"],
    ["Z_QUEENS_1", "Queens"],
    # Z_UNKNOWN_99 intentionally has no row here
], columns=["zone_key", "borough"])


def run_query(fact_taxi_trip: pd.DataFrame, dim_taxi_zone: pd.DataFrame) -> pd.DataFrame:
    """Pandas re-implementation of the CTE + final SELECT in the .sql file."""
    t = fact_taxi_trip.copy()

    # distance_eligible = NOT negative_trip_distance_flag AND trip_distance_miles > 0
    t["distance_eligible"] = (~t["negative_trip_distance_flag"]) & (t["trip_distance_miles"] > 0)
    # fare_eligible = NOT negative_fare_amount_flag
    t["fare_eligible"] = ~t["negative_fare_amount_flag"]

    # LEFT JOIN to dim_taxi_zone on pickup_zone_key = zone_key
    merged = t.merge(
        dim_taxi_zone, left_on="pickup_zone_key", right_on="zone_key", how="left"
    )

    # values only "count" toward fare/distance sums when eligible -- mirrors
    # the SQL's CASE WHEN t.fare_eligible THEN t.fare_amount_usd END pattern
    merged["fare_if_eligible"] = merged["fare_amount_usd"].where(merged["fare_eligible"])
    merged["distance_if_eligible"] = merged["trip_distance_miles"].where(merged["distance_eligible"])

    grouped = merged.groupby("borough", dropna=False).agg(
        trip_count=("trip_count", "sum"),
        total_fare_amount_usd=("fare_if_eligible", "sum"),
        total_trip_distance_miles=("distance_if_eligible", "sum"),
        avg_fare_amount_usd=("fare_if_eligible", "mean"),
        avg_trip_distance_miles=("distance_if_eligible", "mean"),
    ).reset_index()

    grouped["fare_per_mile_usd"] = (
        grouped["total_fare_amount_usd"] / grouped["total_trip_distance_miles"]
    ).round(2)
    grouped["avg_fare_amount_usd"] = grouped["avg_fare_amount_usd"].round(2)
    grouped["avg_trip_distance_miles"] = grouped["avg_trip_distance_miles"].round(3)

    # DENSE_RANK() OVER (ORDER BY fare_per_mile_usd DESC) -- highest $/mi = rank 1
    grouped["fare_per_mile_rank"] = (
        grouped["fare_per_mile_usd"].rank(method="dense", ascending=False).astype(int)
    )

    return grouped.rename(columns={"borough": "pickup_borough"})


def check_dq_gate(result: pd.DataFrame, fact_taxi_trip: pd.DataFrame) -> None:
    """Mirrors the SQL file's closing SELECT (the DQ gate)."""
    analytics_trip_count = result["trip_count"].sum()
    gold_trip_count = fact_taxi_trip["trip_count"].sum()
    result_rows = len(result)
    rows_with_unresolved_borough = result["pickup_borough"].isna().sum()

    print("\n--- DQ gate ---")
    print(f"analytics_trip_count        = {analytics_trip_count}")
    print(f"gold_trip_count              = {gold_trip_count}")
    print(f"result_rows                  = {result_rows}")
    print(f"rows_with_unresolved_borough = {rows_with_unresolved_borough}")

    # trip_count is never filtered in the query, so these two must always match
    assert analytics_trip_count == gold_trip_count, (
        "BUG: trip_count didn't reconcile -- something is filtering rows "
        "before the trip_count SUM, which the query is not supposed to do."
    )

    # in THIS synthetic dataset there is exactly one zone_key (Z_UNKNOWN_99)
    # with no match in dim_taxi_zone, so we expect exactly 1 unresolved row
    assert rows_with_unresolved_borough == 1, (
        f"Expected exactly 1 unresolved-borough row (Z_UNKNOWN_99), "
        f"got {rows_with_unresolved_borough}"
    )
    print("DQ gate checks passed.")


def check_rank_direction(result: pd.DataFrame) -> None:
    """Confirms DENSE_RANK() ORDER BY ... DESC actually gives rank 1 to the
    highest fare_per_mile_usd, not the lowest."""
    top_row = result.loc[result["fare_per_mile_rank"] == 1].iloc[0]
    highest_fare_per_mile = result["fare_per_mile_usd"].max()

    print("\n--- Rank direction check ---")
    print(f"Row ranked #1: {top_row['pickup_borough']} @ ${top_row['fare_per_mile_usd']}/mi")
    print(f"Actual max $/mi in the data: ${highest_fare_per_mile}")

    assert top_row["fare_per_mile_usd"] == highest_fare_per_mile, (
        "BUG: rank 1 is not the highest fare_per_mile_usd -- the ORDER BY "
        "direction in the SQL is backwards (should be DESC)."
    )
    print("Rank direction is correct (DESC -> highest $/mi = rank 1).")


if __name__ == "__main__":
    result = run_query(fact_taxi_trip, dim_taxi_zone)
    print("--- Query result ---")
    print(result.to_string(index=False))

    check_dq_gate(result, fact_taxi_trip)
    check_rank_direction(result)

    print("\nAll sanity checks passed. Safe to commit.")

--- Query result ---
pickup_borough  trip_count  total_fare_amount_usd  total_trip_distance_miles  avg_fare_amount_usd  avg_trip_distance_miles  fare_per_mile_usd  fare_per_mile_rank
      Brooklyn           2                   20.0                       13.0                 20.0                      6.5               1.54                   4
     Manhattan           2                   65.0                        7.0                 32.5                      3.5               9.29                   1
        Queens           3                   57.0                        8.0                 19.0                      8.0               7.12                   2
           NaN           1                   20.0                        4.0                 20.0                      4.0               5.00                   3

--- DQ gate ---
analytics_trip_count        = 8
gold_trip_count              = 8
result_rows                  = 4
rows_with_unresolved_borough = 1
DQ gate checks passed